In [1]:
import os
import re
import json
import math
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)

# =========================
# CẤU HÌNH ĐƯỜNG DẪN
# =========================
PROJECT_DIR = Path.cwd().resolve().parent
SCRIPTS_DIR = PROJECT_DIR / "scripts"
DATA_DIR = PROJECT_DIR / "data_topcv"

RAW_INPUT_PATH = DATA_DIR / "topcv_all_fields_merged_2026-03-16_20-57-23.xlsx"

OUTPUT_DIR = SCRIPTS_DIR / "outputs"
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
REPORT_DIR = OUTPUT_DIR / "reports"

for p in [OUTPUT_DIR, ARTIFACT_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR    :", PROJECT_DIR)
print("SCRIPTS_DIR    :", SCRIPTS_DIR)
print("DATA_DIR       :", DATA_DIR)
print("RAW_INPUT_PATH :", RAW_INPUT_PATH)
print("OUTPUT_DIR     :", OUTPUT_DIR)
print("FILE EXISTS?   :", RAW_INPUT_PATH.exists())

PROJECT_DIR    : D:\TTTN\Project
SCRIPTS_DIR    : D:\TTTN\Project\scripts
DATA_DIR       : D:\TTTN\Project\data_topcv
RAW_INPUT_PATH : D:\TTTN\Project\data_topcv\topcv_all_fields_merged_2026-03-16_20-57-23.xlsx
OUTPUT_DIR     : D:\TTTN\Project\scripts\outputs
FILE EXISTS?   : True


In [2]:
# pd.set_option("display.max_rows", 100)
# pd.set_option("display.max_columns", 200)
# pd.set_option("display.width", 200)
# pd.set_option("display.max_colwidth", 300)

# 1. Load dữ liệu và tiền xử lý cơ bản

In [3]:
# =========================
# HELPER FUNCTIONS
# =========================

EMPTY_TOKENS = {"", "nan", "none", "null", "n/a", "na", "không rõ", "chưa cập nhật"}

def normalize_empty_value(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    if val_str.lower() in EMPTY_TOKENS:
        return None
    return val_str

def first_non_empty(*values):
    for v in values:
        v = normalize_empty_value(v)
        if v is not None:
            return v
    return None

def normalize_unicode(text: str) -> str:
    if text is None:
        return ""
    return unicodedata.normalize("NFC", str(text))

def remove_html(text: str) -> str:
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"&nbsp;?", " ", text)
    return text

def normalize_dash(text: str) -> str:
    return (
        text.replace("•", " - ")
            .replace("▪", " - ")
            .replace("✅", " - ")
            .replace("✔", " - ")
            .replace("★", " - ")
            .replace("●", " - ")
            .replace("−", "-")
            .replace("–", "-")
            .replace("—", "-")
    )

def clean_text_light(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = normalize_unicode(text)
    text = remove_html(text)
    text = normalize_dash(text)

    text = text.replace("\\n", "\n")
    text = re.sub(r"[\r\t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)

    text = re.sub(r"[^\w\sÀ-ỹ\.,;:/\-\+\#\(\)%]", " ", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r" ?\n ?", "\n", text)

    return text.strip()

def clean_text_preserve_structure(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = normalize_unicode(text)
    text = remove_html(text)
    text = normalize_dash(text)

    text = text.replace("\\n", "\n")
    text = re.sub(r"[\r\t]+", " ", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)

    return text.strip()

def clean_text_strict(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = normalize_unicode(text).lower()
    text = remove_html(text)
    text = normalize_dash(text)

    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)
    text = re.sub(r"[^\w\sÀ-ỹ\./\-\+\#]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def safe_str(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    return str(x).strip()

def deduplicate_list(values):
    out, seen = [], set()
    for v in values:
        if not v:
            continue
        if v not in seen:
            seen.add(v)
            out.append(v)
    return out

In [4]:
def load_raw_data(input_path: Path) -> pd.DataFrame:
    ext = input_path.suffix.lower()
    if ext == ".xlsx":
        df = pd.read_excel(input_path)
    elif ext == ".csv":
        df = pd.read_csv(input_path)
    else:
        raise ValueError(f"Định dạng file không hỗ trợ: {ext}")

    print(f"[INFO] Loaded raw data: {df.shape[0]} rows x {df.shape[1]} cols")
    return df

raw_df = load_raw_data(RAW_INPUT_PATH)
display(raw_df.head(3))
print("\nColumns:\n", raw_df.columns.tolist())

[INFO] Loaded raw data: 325 rows x 33 cols


,job_url,source_field_name,field_count,title,detail_title,company_name,company_name_full,company_url,company_url_from_job,salary_list,detail_salary,address_list,detail_location,exp_list,detail_experience,deadline,tags,job_level,education_level,job_quantity,employment_type,working_addresses,working_times,desc_mota,desc_yeucau,desc_quyenloi,company_website,company_scale_from_job,company_scale,company_field_from_job,company_address_from_job,company_address,company_description
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Data Analyst,1,Junior FP&A Analyst - Hồ Chí Minh,Junior FP&A Analyst - Hồ Chí Minh,Bee Logistics Corporation,Bee Logistics Corporation,https://www.topcv.vn/brand/beeogisticscorporation?id=2450,https://www.topcv.vn/brand/beeogisticscorporation?id=2450,12 - 20 triệu,12 - 20 triệu,Hồ Chí Minh (mới),Hồ Chí Minh,2 năm,2 năm,27/03/2026,Chuyên môn Data Analyst; Tài chính; Kế toán,Nhân viên,Đại Học trở lên,1 người,Toàn thời gian,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00) Nghỉ 2 buổi sáng Thứ 7/tháng\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00)\nNghỉ 2 buổi sáng Thứ 7/tháng,"– Overseeing all financial planning, reporting & analysis for the Bee office. – Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...",– At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. – Bachelor's degree in Finance/Accounting...,"– Competitive salary according to personal experience and ability – Lunch allowance, phone allowance – Salary increase and annual bonus on holidays (2/9, 1/5, Mid-Autumn Festival, 1/6, New Year, L...",http://www.beelogistics.com/,NaN,500-1000,NaN,NaN,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu, Ward 2, Tan Binh District, Ho Chi Minh City, Viet Nam Addr: 10th Floor, 789 Tower, 147 Hoang Quoc Viet, Nghia Do Ward, Cau Giay District, Hanoi City...","Xuất phát với ước mơ xây dựng một doanh nghiệp Việt, có thể cung cấp, phát triển dịch vụ logistics toàn cầu, tin cậy dựa trên chất lượng dịch vụ, con người và công nghệ, cùng niềm đam mê với nghề ..."
1,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,Data Analyst,1,Data Governance Specialist,Data Governance Specialist,EDUPIA,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://www.topcv.vn/brand/educa?id=17270,20 - 30 triệu,20 - 30 triệu,Hà Nội,NaN,2 năm,2 năm,Còn 17 ngày để ứng tuyển,Chuyên môn Data Analyst,Nhân viên,Đại Học trở lên,1 người,Toàn thời gian,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00)\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00),"− Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. − Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","Mức lương thỏa thuận theo năng lực. Thử việc hưởng 100% lương trong 2 tháng. Thưởng Lễ/Tết, thưởng hiệu quả làm việc: theo quy định của Công ty. Chính sách đánh giá điều chỉnh lương hàng năm minh ...",https://edupia.vn/,NaN,500-1000,NaN,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Tower, số 61 Ngụy Như Kon Tum, Thanh Xuân, Hà Nội. Chi nhánh: Tầng 6, Tòa nhà Báo Sinh Viên - Hoa Học Trò, Yên Hòa, Cầu Giấy, Hà Nội","Được thành lập


Columns:
 ['job_url', 'source_field_name', 'field_count', 'title', 'detail_title', 'company_name', 'company_name_full', 'company_url', 'company_url_from_job', 'salary_list', 'detail_salary', 'address_list', 'detail_location', 'exp_list', 'detail_experience', 'deadline', 'tags', 'job_level', 'education_level', 'job_quantity', 'employment_type', 'working_addresses', 'working_times', 'desc_mota', 'desc_yeucau', 'desc_quyenloi', 'company_website', 'company_scale_from_job', 'company_scale', 'company_field_from_job', 'company_address_from_job', 'company_address', 'company_description']


In [5]:
def merge_semantic_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()

    out["job_url"] = df.get("job_url")
    out["source_field_name"] = df.get("source_field_name")
    out["field_count"] = df.get("field_count")

    out["job_title_raw"] = [
        first_non_empty(a, b) for a, b in zip(df.get("detail_title"), df.get("title"))
    ]

    out["salary_raw"] = [
        first_non_empty(a, b) for a, b in zip(df.get("detail_salary"), df.get("salary_list"))
    ]
    out["location_raw"] = [
        first_non_empty(a, b) for a, b in zip(df.get("detail_location"), df.get("address_list"))
    ]
    out["working_addresses_raw"] = df.get("working_addresses")
    out["working_times_raw"] = df.get("working_times")
    out["experience_raw"] = [
        first_non_empty(a, b) for a, b in zip(df.get("detail_experience"), df.get("exp_list"))
    ]
    out["tags_raw"] = df.get("tags")

    out["job_level_raw"] = df.get("job_level")
    out["education_level_raw"] = df.get("education_level")
    out["employment_type_raw"] = df.get("employment_type")
    out["job_quantity"] = df.get("job_quantity")
    out["deadline_raw"] = df.get("deadline")

    out["description_raw"] = df.get("desc_mota")
    out["requirements_raw"] = df.get("desc_yeucau")
    out["benefits_raw"] = df.get("desc_quyenloi")

    out["company_name_raw"] = [
        first_non_empty(a, b) for a, b in zip(df.get("company_name_full"), df.get("company_name"))
    ]
    out["company_url"] = [
        first_non_empty(a, b) for a, b in zip(df.get("company_url_from_job"), df.get("company_url"))
    ]
    out["company_website_raw"] = df.get("company_website")
    out["company_scale_raw"] = [
        first_non_empty(a, b) for a, b in zip(df.get("company_scale_from_job"), df.get("company_scale"))
    ]
    out["company_field_raw"] = df.get("company_field_from_job")
    out["company_address_raw"] = [
        first_non_empty(a, b) for a, b in zip(df.get("company_address_from_job"), df.get("company_address"))
    ]
    out["company_description_raw"] = df.get("company_description")

    print(f"[INFO] After merging: {out.shape[0]} rows x {out.shape[1]} cols")
    return out

df = merge_semantic_columns(raw_df)
display(df.head(3))
df.info()

[INFO] After merging: 325 rows x 25 cols


,job_url,source_field_name,field_count,job_title_raw,salary_raw,location_raw,working_addresses_raw,working_times_raw,experience_raw,tags_raw,job_level_raw,education_level_raw,employment_type_raw,job_quantity,deadline_raw,description_raw,requirements_raw,benefits_raw,company_name_raw,company_url,company_website_raw,company_scale_raw,company_field_raw,company_address_raw,company_description_raw
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Data Analyst,1,Junior FP&A Analyst - Hồ Chí Minh,12 - 20 triệu,Hồ Chí Minh,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00) Nghỉ 2 buổi sáng Thứ 7/tháng\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00)\nNghỉ 2 buổi sáng Thứ 7/tháng,2 năm,Chuyên môn Data Analyst; Tài chính; Kế toán,Nhân viên,Đại Học trở lên,Toàn thời gian,1 người,27/03/2026,"– Overseeing all financial planning, reporting & analysis for the Bee office. – Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...",– At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. – Bachelor's degree in Finance/Accounting...,"– Competitive salary according to personal experience and ability – Lunch allowance, phone allowance – Salary increase and annual bonus on holidays (2/9, 1/5, Mid-Autumn Festival, 1/6, New Year, L...",Bee Logistics Corporation,https://www.topcv.vn/brand/beeogisticscorporation?id=2450,http://www.beelogistics.com/,500-1000,NaN,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu, Ward 2, Tan Binh District, Ho Chi Minh City, Viet Nam Addr: 10th Floor, 789 Tower, 147 Hoang Quoc Viet, Nghia Do Ward, Cau Giay District, Hanoi City...","Xuất phát với ước mơ xây dựng một doanh nghiệp Việt, có thể cung cấp, phát triển dịch vụ logistics toàn cầu, tin cậy dựa trên chất lượng dịch vụ, con người và công nghệ, cùng niềm đam mê với nghề ..."
1,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,Data Analyst,1,Data Governance Specialist,20 - 30 triệu,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00)\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00),2 năm,Chuyên môn Data Analyst,Nhân viên,Đại Học trở lên,Toàn thời gian,1 người,Còn 17 ngày để ứng tuyển,"− Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. − Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","Mức lương thỏa thuận theo năng lực. Thử việc hưởng 100% lương trong 2 tháng. Thưởng Lễ/Tết, thưởng hiệu quả làm việc: theo quy định của Công ty. Chính sách đánh giá điều chỉnh lương hàng năm minh ...",EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://edupia.vn/,500-1000,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Tower, số 61 Ngụy Như Kon Tum, Thanh Xuân, Hà Nội. Chi nhánh: Tầng 6, Tòa nhà Báo Sinh Viên - Hoa Học Trò, Yên Hòa, Cầu Giấy, Hà Nội","Được thành lập năm 2018, Edupia là Edtech lớn nhất Việt Nam về Tiếng Anh Online cho trẻ em, đồng thời là Top 50 Edtech Nổi bật nhất Đông Nam Á năm 2022 & 2023. Qua nhiều năm phát triển, Edupia đến..."
2,https://www.topcv.vn/brand/educa/tuyen-dung/project-manager-du-an-ai-hub-j2067747.html,AI Engineer,1,Project Manager Dự Án AI HUB,30 - 35 triệu,Hà Nội,"(đã được cập 

<class 'pandas.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   job_url                  325 non-null    str  
 1   source_field_name        325 non-null    str  
 2   field_count              325 non-null    int64
 3   job_title_raw            325 non-null    str  
 4   salary_raw               325 non-null    str  
 5   location_raw             325 non-null    str  
 6   working_addresses_raw    325 non-null    str  
 7   working_times_raw        294 non-null    str  
 8   experience_raw           325 non-null    str  
 9   tags_raw                 325 non-null    str  
 10  job_level_raw            325 non-null    str  
 11  education_level_raw      325 non-null    str  
 12  employment_type_raw      325 non-null    str  
 13  job_quantity             325 non-null    str  
 14  deadline_raw             325 non-null    str  
 15  description_raw  

# 2. Audit dữ liệu và làm sạch

In [6]:
df_clean = df.copy()

print(f"[INFO] df_raw shape   : {df.shape}")
print(f"[INFO] df_clean shape : {df_clean.shape}")

[INFO] df_raw shape   : (325, 25)
[INFO] df_clean shape : (325, 25)


In [7]:
def detect_language_type(text: str) -> str:
    text = safe_str(text)
    if not text:
        return "unknown"

    has_vietnamese = bool(re.search(r"[À-ỹđĐ]", text))
    has_ascii_words = bool(re.search(r"[A-Za-z]", text))

    if has_vietnamese and has_ascii_words:
        return "mixed"
    if has_vietnamese:
        return "vi"
    if has_ascii_words:
        return "en"
    return "unknown"

df_clean["job_title_raw_norm_for_dup"] = df_clean["job_title_raw"].apply(
    lambda x: clean_text_strict(x)
)
df_clean["company_name_raw_norm_for_dup"] = df_clean["company_name_raw"].apply(
    lambda x: clean_text_strict(x)
)

df_clean["is_duplicate_by_url"] = df_clean["job_url"].duplicated(keep=False)
df_clean["is_duplicate_soft"] = (
    df_clean
    .assign(
        dup_key=(
            df_clean["company_name_raw_norm_for_dup"].fillna("") + " || " +
            df_clean["job_title_raw_norm_for_dup"].fillna("")
        )
    )["dup_key"]
    .duplicated(keep=False)
)

df_clean["language_type"] = (
    df_clean["job_title_raw"].fillna("") + " " +
    df_clean["requirements_raw"].fillna("")
).apply(detect_language_type)

df_clean["has_minimum_content"] = (
    df_clean["job_title_raw"].fillna("").str.len().ge(3) &
    (
        df_clean["description_raw"].fillna("").str.len().ge(30) |
        df_clean["requirements_raw"].fillna("").str.len().ge(30)
    )
)

audit_report = pd.DataFrame({
    "n_rows": [len(df_clean)],
    "dup_by_url_ratio": [round(df_clean["is_duplicate_by_url"].mean(), 4)],
    "dup_soft_ratio": [round(df_clean["is_duplicate_soft"].mean(), 4)],
    "has_minimum_content_ratio": [round(df_clean["has_minimum_content"].mean(), 4)],
})

display(audit_report)

missing_report = (
    df_clean.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_ratio")
    .reset_index()
    .rename(columns={"index": "column"})
)

display(missing_report.head(20))

,n_rows,dup_by_url_ratio,dup_soft_ratio,has_minimum_content_ratio
0,325,0.0,0.0615,1.0


,column,missing_ratio
0,company_website_raw,0.169231
1,working_times_raw,0.095385
2,company_field_raw,0.080000
3,company_description_raw,0.027692
4,job_title_raw,0.000000
5,source_field_name,0.000000
6,job_url,0.000000
7,working_addresses_raw,0.000000
8,experience_raw,0.000000
9,tags_raw,0.000000


In [8]:
raw_text_cols = [
    "job_title_raw",
    "salary_raw",
    "location_raw",
    "working_addresses_raw",
    "working_times_raw",
    "experience_raw",
    "tags_raw",
    "job_level_raw",
    "education_level_raw",
    "employment_type_raw",
    "deadline_raw",
    "description_raw",
    "requirements_raw",
    "benefits_raw",
    "company_name_raw",
    "company_url",
    "company_website_raw",
    "company_scale_raw",
    "company_field_raw",
    "company_address_raw",
    "company_description_raw"
]

for col in raw_text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(normalize_empty_value)

print("[INFO] Đã chuẩn hóa giá trị rỗng cho các cột raw.")

[INFO] Đã chuẩn hóa giá trị rỗng cho các cột raw.


In [9]:
light_clean_cols = [
    "job_title_raw",
    "description_raw",
    "requirements_raw",
    "benefits_raw",
    "company_description_raw",
    "working_times_raw",
    "working_addresses_raw",
    "company_address_raw",
    "company_name_raw",
    "tags_raw"
]

for col in light_clean_cols:
    if col in df_clean.columns:
        df_clean[col.replace("_raw", "_clean_light")] = df_clean[col].apply(clean_text_light)
        df_clean[col.replace("_raw", "_clean_struct")] = df_clean[col].apply(clean_text_preserve_structure)

print("[INFO] Đã tạo xong các cột *_clean_light và *_clean_struct")

[INFO] Đã tạo xong các cột *_clean_light và *_clean_struct


In [10]:
strict_clean_cols = [
    "job_title_raw",
    "salary_raw",
    "location_raw",
    "experience_raw",
    "education_level_raw",
    "employment_type_raw",
    "job_level_raw",
    "tags_raw",
    "company_field_raw",
    "working_times_raw",
    "working_addresses_raw",
    "requirements_raw",
    "description_raw"
]

for col in strict_clean_cols:
    if col in df_clean.columns:
        df_clean[col.replace("_raw", "_clean_strict")] = df_clean[col].apply(clean_text_strict)

print("[INFO] Đã tạo xong các cột *_clean_strict")

[INFO] Đã tạo xong các cột *_clean_strict


In [11]:
clean_cols_created = [
    c for c in df_clean.columns
    if c.endswith("_clean_light") or c.endswith("_clean_strict") or c.endswith("_clean_struct")
]

print(f"[INFO] Số cột clean đã tạo: {len(clean_cols_created)}")
print(clean_cols_created)

[INFO] Số cột clean đã tạo: 33
['job_title_clean_light', 'job_title_clean_struct', 'description_clean_light', 'description_clean_struct', 'requirements_clean_light', 'requirements_clean_struct', 'benefits_clean_light', 'benefits_clean_struct', 'company_description_clean_light', 'company_description_clean_struct', 'working_times_clean_light', 'working_times_clean_struct', 'working_addresses_clean_light', 'working_addresses_clean_struct', 'company_address_clean_light', 'company_address_clean_struct', 'company_name_clean_light', 'company_name_clean_struct', 'tags_clean_light', 'tags_clean_struct', 'job_title_clean_strict', 'salary_clean_strict', 'location_clean_strict', 'experience_clean_strict', 'education_level_clean_strict', 'employment_type_clean_strict', 'job_level_clean_strict', 'tags_clean_strict', 'company_field_clean_strict', 'working_times_clean_strict', 'working_addresses_clean_strict', 'requirements_clean_strict', 'description_clean_strict']


In [12]:
preview_cols = [
    "job_title_raw", "job_title_clean_light", "job_title_clean_strict",
    "requirements_raw", "requirements_clean_light", "requirements_clean_struct", "requirements_clean_strict",
    "description_raw", "description_clean_light", "description_clean_struct", "description_clean_strict"
]
preview_cols = [c for c in preview_cols if c in df_clean.columns]

display(df_clean[preview_cols].head(3))

,job_title_raw,job_title_clean_light,job_title_clean_strict,requirements_raw,requirements_clean_light,requirements_clean_struct,requirements_clean_strict,description_raw,description_clean_light,description_clean_struct,description_clean_strict
0,Junior FP&A Analyst - Hồ Chí Minh,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst - hồ chí minh,– At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. – Bachelor's degree in Finance/Accounting...,"- At least 2 year experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor s degree in Finance/Accounting,...",- At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor's degree in Finance/Accounting...,- at least 2 year experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - bachelor s degree in finance/accounting ...,"– Overseeing all financial planning, reporting & analysis for the Bee office. – Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...","- Overseeing all financial planning, reporting analysis for the Bee office. - Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-quali...","- Overseeing all financial planning, reporting & analysis for the Bee office. - Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...",- overseeing all financial planning reporting analysis for the bee office. - track management reports focusing on budget/estimate/actuals to ensure those are delivered on time and with high-qualit...
1,Data Governance Specialist,Data Governance Specialist,data governance specialist,"− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...",- đã tốt nghiệp đh trở lên ưu tiên các chuyên ngành về công nghệ thông tin/ hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực data governance/ da...,"− Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. − Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","- Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. - Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","- Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. - Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...",- xây dựng triển khai và duy trì khung data governance cho toàn hệ thống dữ liệu của công ty. - xây dựng chính sách tiêu chuẩn quy định về quản lý dữ liệu data quality data profiling data security...
2,Project Manager Dự Án AI HUB,Project Manager Dự Án AI HUB,project manager dự án ai hub,"Tối thiểu 2–3 năm kinh nghiệm ở vị trí Product Manager, Project Manager hoặc Operations Manager. Ưu tiên ứng viên từng tham gia các dự án automation, AI, data analytics hoặc chuyển đổi số. Ưu tiên...","Tối thiểu 2-3 năm kinh nghiệm ở vị trí Product Manager, P

In [13]:
empty_after_clean_df = pd.DataFrame({
    "column": preview_cols,
    "empty_ratio": [
        df_clean[col].fillna("").astype(str).str.strip().eq("").mean()
        for col in preview_cols
    ]
}).sort_values("empty_ratio", ascending=False)

display(empty_after_clean_df)

,column,empty_ratio
0,job_title_raw,0.0
1,job_title_clean_light,0.0
2,job_title_clean_strict,0.0
3,requirements_raw,0.0
4,requirements_clean_light,0.0
5,requirements_clean_struct,0.0
6,requirements_clean_strict,0.0
7,description_raw,0.0
8,description_clean_light,0.0
9,description_clean_struct,0.0


In [14]:
length_check_cols = [
    "description_clean_light",
    "requirements_clean_light",
    "benefits_clean_light",
    "company_description_clean_light"
]
length_check_cols = [c for c in length_check_cols if c in df_clean.columns]

for col in length_check_cols:
    df_clean[f"{col}_len"] = df_clean[col].fillna("").str.len()

display(
    df_clean[[f"{c}_len" for c in length_check_cols]].describe().T
)

,count,mean,std,min,25%,50%,75%,max
description_clean_light_len,325.0,1018.560000,780.434020,149.0,535.0,821.0,1209.0,6034.0
requirements_clean_light_len,325.0,858.947692,622.290418,118.0,485.0,708.0,1084.0,5829.0
benefits_clean_light_len,325.0,728.889231,435.602067,90.0,388.0,614.0,1037.0,3461.0
company_description_clean_light_len,325.0,932.972308,721.431706,0.0,437.0,756.0,1284.0,3699.0


In [15]:
clean_checkpoint_path = ARTIFACT_DIR / "jobs_cleaning_checkpoint.parquet"
df_clean.to_parquet(clean_checkpoint_path, index=False)
print(f"[INFO] Saved checkpoint: {clean_checkpoint_path}")

[INFO] Saved checkpoint: D:\TTTN\Project\scripts\outputs\artifacts\jobs_cleaning_checkpoint.parquet


# 3. Normalize metadata

In [16]:
TITLE_NOISE_PATTERNS = [
    r"\b(lương|thu nhập|up to|mức lương|deal lương|thỏa thuận|thoả thuận)\b.*$",
    r"\b(tại|ở)\s+(hà nội|ha noi|hồ chí minh|ho chi minh|đà nẵng|da nang|remote|hybrid).*$",
    r"\b(đi làm ngay|urgent|hot|gấp|nhận việc ngay)\b.*$",
    r"\b\d+\+?\s*năm kinh nghiệm\b.*$",
    r"\(.*?\)",
    r"\[.*?\]",
    r"\s*\|\s*.*$"
]

TITLE_SYNONYM_MAP = {
    "chuyên viên phân tích dữ liệu": "data analyst",
    "nhân viên phân tích dữ liệu": "data analyst",
    "data analyst": "data analyst",
    "business analyst": "business analyst",
    "chuyên viên phân tích nghiệp vụ": "business analyst",
    "data engineer": "data engineer",
    "data scientist": "data scientist",
    "machine learning engineer": "machine learning engineer",
    "ml engineer": "machine learning engineer",
    "ai engineer": "ai engineer",
    "backend developer": "backend developer",
    "backend engineer": "backend developer",
    "frontend developer": "frontend developer",
    "fullstack developer": "fullstack developer",
    "full-stack developer": "fullstack developer",
}

JOB_FAMILY_RULES = {
    "data_analytics": ["data analyst", "business analyst", "bi analyst", "analytics"],
    "data_engineering": ["data engineer", "etl developer", "data warehouse"],
    "data_science_ml": ["data scientist", "machine learning engineer", "ai engineer", "ml engineer"],
    "software_engineering": ["backend developer", "frontend developer", "fullstack developer"]
}

def normalize_job_title(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = clean_text_light(text)

    for pattern in TITLE_NOISE_PATTERNS:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()

    text = re.sub(r"\s+", " ", text).strip(" -|,")
    text_lower = text.lower()

    return TITLE_SYNONYM_MAP.get(text_lower, text_lower)

def infer_job_family(job_title_canonical: str) -> str:
    jt = safe_str(job_title_canonical).lower()
    if not jt:
        return "unknown"

    for family, patterns in JOB_FAMILY_RULES.items():
        if any(p in jt for p in patterns):
            return family
    return "other"

def infer_seniority_from_title(text: str) -> str:
    t = clean_text_strict(text)
    if not t:
        return "unknown"

    if any(k in t for k in ["intern", "thực tập"]):
        return "intern"
    if any(k in t for k in ["fresher", "junior", "nhân viên", "chuyên viên"]):
        return "junior"
    if any(k in t for k in ["senior"]):
        return "senior"
    if any(k in t for k in ["team lead", "leader", "trưởng nhóm", "lead"]):
        return "lead"
    if any(k in t for k in ["manager", "quản lý", "head"]):
        return "manager"
    if "director" in t:
        return "director"
    return "unknown"

In [17]:
df_clean["job_title_display"] = df_clean["job_title_raw"].apply(clean_text_light)
df_clean["job_title_canonical"] = df_clean["job_title_raw"].apply(normalize_job_title)
df_clean["job_family"] = df_clean["job_title_canonical"].apply(infer_job_family)
df_clean["seniority_from_title"] = df_clean["job_title_raw"].apply(infer_seniority_from_title)

display(
    df_clean[
        ["job_title_raw", "job_title_display", "job_title_canonical", "job_family", "seniority_from_title"]
    ].head(10)
)

,job_title_raw,job_title_display,job_title_canonical,job_family,seniority_from_title
0,Junior FP&A Analyst - Hồ Chí Minh,Junior FP A Analyst - Hồ Chí Minh,junior fp a analyst - hồ chí minh,other,junior
1,Data Governance Specialist,Data Governance Specialist,data governance specialist,other,unknown
2,Project Manager Dự Án AI HUB,Project Manager Dự Án AI HUB,project manager dự án ai hub,other,manager
3,AI Engineer,AI Engineer,ai engineer,data_science_ml,unknown
4,AI Engineer,AI Engineer,ai engineer,data_science_ml,unknown
5,Data Analyst,Data Analyst,data analyst,data_analytics,unknown
6,Data Engineer,Data Engineer,data engineer,data_engineering,unknown
7,Fresher Data Engineer,Fresher Data Engineer,fresher data engineer,data_engineering,junior
8,Data Analyst Teamleader (Collection Analytics),Data Analyst Teamleader (Collection Analytics),data analyst teamleader,data_analytics,lead
9,Data Analyst Teamleader (Collection Analytics) - Thu Nhập 50 Triệu Tại Hồ Chí Minh,Data Analyst Teamleader (Collection Analytics) - Thu Nhập 50 Triệu Tại Hồ Chí Minh,data analyst teamleader,data_analytics,lead


In [18]:
VIETNAM_CITIES = sorted({
    "hà nội", "ha noi", "hồ chí minh", "ho chi minh", "đà nẵng", "da nang",
    "hải phòng", "hai phong", "cần thơ", "can tho", "huế", "hue",
    "an giang", "bắc ninh", "bac ninh", "cà mau", "ca mau", "cao bằng", "cao bang",
    "đắk lắk", "dak lak", "điện biên", "dien bien", "đồng nai", "dong nai",
    "đồng tháp", "dong thap", "gia lai", "hà tĩnh", "ha tinh", "hưng yên", "hung yen",
    "khánh hòa", "khanh hoa", "lai châu", "lai chau", "lâm đồng", "lam dong",
    "lạng sơn", "lang son", "lào cai", "lao cai", "nghệ an", "nghe an",
    "ninh bình", "ninh binh", "phú thọ", "phu tho", "quảng ngãi", "quang ngai",
    "quảng ninh", "quang ninh", "quảng trị", "quang tri", "sơn la", "son la",
    "tây ninh", "tay ninh", "thái nguyên", "thai nguyen", "thanh hóa", "thanh hoa",
    "tuyên quang", "tuyen quang", "vĩnh long", "vinh long"
})

def infer_work_mode(*texts) -> str:
    merged = " ".join(clean_text_strict(t) for t in texts if normalize_empty_value(t))
    if not merged:
        return "unknown"
    if "remote" in merged or "làm việc từ xa" in merged:
        return "remote"
    if "hybrid" in merged:
        return "hybrid"
    if "onsite" in merged or "tại văn phòng" in merged:
        return "onsite"
    return "unknown"

In [19]:
DISTRICT_PATTERN = re.compile(r'(?i)\b(quận|huyện|thị xã|thành phố)\s+[^\n,;()]+')

def detect_city_from_text(text: str, city_list=None):
    city_list = city_list or VIETNAM_CITIES
    t = clean_text_strict(text)
    if not t:
        return None
    for city in city_list:
        if city in t:
            return city
    return None

def parse_working_address(raw_text: str) -> dict:
    text = clean_text_preserve_structure(raw_text)
    if not text:
        return {
            "location_city": None,
            "location_district": None,
            "working_address_clean": "",
        }

    city = detect_city_from_text(text)
    district_match = DISTRICT_PATTERN.search(text)
    district = district_match.group(0).strip() if district_match else None

    text = re.sub(r"\s+", " ", text).strip(" -,")

    return {
        "location_city": city,
        "location_district": district,
        "working_address_clean": text,
    }

def normalize_location(location_raw: str, working_addresses_raw: str) -> str:
    loc = clean_text_light(location_raw)
    addr_info = parse_working_address(working_addresses_raw)

    if loc:
        loc_city = detect_city_from_text(loc)
        if loc_city:
            return loc_city

    if addr_info["location_city"]:
        return addr_info["location_city"]

    return loc.lower() if loc else ""

In [20]:
addr_parsed = df_clean["working_addresses_raw"].apply(parse_working_address)

df_clean["working_address_clean"] = addr_parsed.apply(lambda x: x["working_address_clean"])
df_clean["location_city"] = addr_parsed.apply(lambda x: x["location_city"])
df_clean["location_district"] = addr_parsed.apply(lambda x: x["location_district"])

df_clean["location_norm"] = df_clean.apply(
    lambda row: normalize_location(row.get("location_raw"), row.get("working_addresses_raw")),
    axis=1
)

df_clean["work_mode"] = df_clean.apply(
    lambda row: infer_work_mode(
        row.get("job_title_raw"),
        row.get("working_times_raw"),
        row.get("description_raw"),
        row.get("requirements_raw")
    ),
    axis=1
)

display(
    df_clean[
        ["location_raw", "working_addresses_raw", "location_city", "location_district", "location_norm", "work_mode"]
    ].head(10)
)

,location_raw,working_addresses_raw,location_city,location_district,location_norm,work_mode
0,Hồ Chí Minh,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)",hồ chí minh,huyện cũ tương ứng để dễ dàng tra cứu,hồ chí minh,unknown
1,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...",hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,unknown
2,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2,3,5,6 Tòa Comatce Tower - 61 Ngụy Như Kon Tum, Thanh Xuân, Phường Thanh Xuân (qu...",hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,unknown
3,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: FPT Tower, số 10 Phạm Văn Bạch, Phường Cầu Giấy (quận Cầu Giấy cũ)",hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,unknown
4,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Landmark 72, Keangnam Phạm Hùng, Phường Yên Hòa (quận Cầu Giấy cũ)",hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,unknown
5,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Keangnam Landmark 72, E6 Phạm Hùng, Phường Yên Hòa (quận Cầu Giấy cũ)",hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,unknown
6,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Keangnam Landmark 72, Phường Yên Hòa (quận Cầu Giấy cũ)",hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,unknown
7,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Khu Công nghệ cao Hòa Lạc, Xã Hòa Lạc (huyện Thạch Thất cũ) - Hà Nội: FPT Tower, số 10 ...",hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,unknown
8,Hồ Chí Minh,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: 20 Cộng Hòa, Phường Bảy Hiền (quận Tân Bình cũ)",hồ chí minh,huyện cũ tương ứng để dễ dàng tra cứu,hồ chí minh,unknown
9,Hồ Chí Minh,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: 20 Cộng Hòa, Phường Bảy Hiền (quận Tân Bình cũ)",hồ chí minh,huyện cũ tương ứng để dễ dàng tra cứu,hồ chí minh,unknown


In [21]:
display(df_clean["location_norm"].value_counts(dropna=False).head(20))
display(df_clean["work_mode"].value_counts(dropna=False))

location_norm
hà nội         237
hồ chí minh     72
đà nẵng          8
hưng yên         3
cần thơ          2
bắc ninh         1
vĩnh long        1
hải phòng        1
Name: count, dtype: int64

work_mode
unknown    303
remote      11
onsite       7
hybrid       4
Name: count, dtype: int64

In [22]:
def clean_salary(raw: str) -> str:
    text = normalize_empty_value(raw)
    if text is None:
        return ""

    text = normalize_unicode(text)
    text = remove_html(text)
    text = text.lower()
    text = text.replace("thoả", "thỏa")
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s*-\s*", " - ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def _to_number(num_text: str):
    try:
        return float(num_text.replace(",", "."))
    except:
        return None

def parse_salary_range(raw: str) -> dict:
    text = clean_salary(raw)
    if not text:
        return {
            "salary_clean": "",
            "salary_min_value": None,
            "salary_max_value": None,
            "salary_currency": "unknown",
            "salary_period": "unknown",
            "salary_type": "unknown",
            "salary_is_negotiable": 0,
            "salary_min_vnd_month": None,
            "salary_max_vnd_month": None,
        }

    currency = "vnd"
    if "usd" in text or "$" in text:
        currency = "usd"

    period = "month"
    if "/năm" in text or "năm" in text:
        period = "year"
    elif "/ngày" in text or "ngày" in text:
        period = "day"
    elif "/giờ" in text or "giờ" in text:
        period = "hour"

    negotiable = int(any(k in text for k in ["thỏa thuận", "thoả thuận", "deal lương", "negotiable"]))

    salary_type = "unknown"
    salary_min = None
    salary_max = None

    range_match = re.search(r"(\d+[.,]?\d*)\s*-\s*(\d+[.,]?\d*)", text)
    if range_match:
        salary_min = _to_number(range_match.group(1))
        salary_max = _to_number(range_match.group(2))
        salary_type = "range"
    else:
        up_to_match = re.search(r"(up to|tới|đến)\s*(\d+[.,]?\d*)", text)
        above_match = re.search(r"(trên|from|từ)\s*(\d+[.,]?\d*)", text)
        single_match = re.search(r"(\d+[.,]?\d*)", text)

        if up_to_match:
            salary_max = _to_number(up_to_match.group(2))
            salary_type = "upper_bound"
        elif above_match:
            salary_min = _to_number(above_match.group(2))
            salary_type = "lower_bound"
        elif single_match:
            salary_min = _to_number(single_match.group(1))
            salary_max = salary_min
            salary_type = "fixed"

    # scale units
    multiplier = 1
    if "triệu" in text:
        multiplier = 1_000_000
    elif "k" in text:
        multiplier = 1_000
    elif currency == "usd":
        multiplier = 1

    salary_min_value = salary_min * multiplier if salary_min is not None else None
    salary_max_value = salary_max * multiplier if salary_max is not None else None

    # normalize to VND / month
    usd_to_vnd = 25000
    def to_vnd_month(x):
        if x is None:
            return None
        val = x * usd_to_vnd if currency == "usd" else x
        if period == "year":
            return val / 12
        if period == "day":
            return val * 22
        if period == "hour":
            return val * 8 * 22
        return val

    return {
        "salary_clean": text,
        "salary_min_value": salary_min_value,
        "salary_max_value": salary_max_value,
        "salary_currency": currency,
        "salary_period": period,
        "salary_type": salary_type,
        "salary_is_negotiable": negotiable,
        "salary_min_vnd_month": to_vnd_month(salary_min_value),
        "salary_max_vnd_month": to_vnd_month(salary_max_value),
    }

In [23]:
salary_examples = pd.Series([
    "12 - 20 triệu",
    "Thoả thuận",
    "Up to 30 triệu",
    "1000 USD",
    "300 triệu/năm",
    "Trên 20 triệu"
])

display(salary_examples.apply(parse_salary_range).apply(pd.Series))

,salary_clean,salary_min_value,salary_max_value,salary_currency,salary_period,salary_type,salary_is_negotiable,salary_min_vnd_month,salary_max_vnd_month
0,12 - 20 triệu,12000000.0,20000000.0,vnd,month,range,0,12000000.0,20000000.0
1,thỏa thuận,NaN,NaN,vnd,month,unknown,1,NaN,NaN
2,up to 30 triệu,NaN,30000000.0,vnd,month,upper_bound,0,NaN,30000000.0
3,1000 usd,1000.0,1000.0,usd,month,fixed,0,25000000.0,25000000.0
4,300 triệu/năm,300000000.0,300000000.0,vnd,year,fixed,0,25000000.0,25000000.0
5,trên 20 triệu,20000000.0,NaN,vnd,month,lower_bound,0,20000000.0,NaN


In [24]:
salary_parsed = df_clean["salary_raw"].apply(parse_salary_range)

salary_cols = [
    "salary_clean",
    "salary_min_value",
    "salary_max_value",
    "salary_currency",
    "salary_period",
    "salary_type",
    "salary_is_negotiable",
    "salary_min_vnd_month",
    "salary_max_vnd_month",
]

for col in salary_cols:
    df_clean[col] = salary_parsed.apply(lambda x: x[col])

display(
    df_clean[
        ["salary_raw"] + salary_cols
    ].head(10)
)

,salary_raw,salary_clean,salary_min_value,salary_max_value,salary_currency,salary_period,salary_type,salary_is_negotiable,salary_min_vnd_month,salary_max_vnd_month
0,12 - 20 triệu,12 - 20 triệu,12000000.0,20000000.0,vnd,month,range,0,12000000.0,20000000.0
1,20 - 30 triệu,20 - 30 triệu,20000000.0,30000000.0,vnd,month,range,0,20000000.0,30000000.0
2,30 - 35 triệu,30 - 35 triệu,30000000.0,35000000.0,vnd,month,range,0,30000000.0,35000000.0
3,Thoả thuận,thỏa thuận,NaN,NaN,vnd,month,unknown,1,NaN,NaN
4,Thoả thuận,thỏa thuận,NaN,NaN,vnd,month,unknown,1,NaN,NaN
5,Thoả thuận,thỏa thuận,NaN,NaN,vnd,month,unknown,1,NaN,NaN
6,Thoả thuận,thỏa thuận,NaN,NaN,vnd,month,unknown,1,NaN,NaN
7,6 - 9 triệu,6 - 9 triệu,6000000.0,9000000.0,vnd,month,range,0,6000000.0,9000000.0
8,Thoả thuận,thỏa thuận,NaN,NaN,vnd,month,unknown,1,NaN,NaN
9,Thoả thuận,thỏa thuận,NaN,NaN,vnd,month,unknown,1,NaN,NaN


In [25]:
def clean_experience(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""
    text = clean_text_strict(text)
    return text

def parse_experience_range(raw: str) -> dict:
    text = clean_experience(raw)
    if not text:
        return {
            "experience_clean": "",
            "experience_min_years": None,
            "experience_max_years": None,
            "experience_type": "unknown",
        }

    if "không yêu cầu" in text or "chưa có kinh nghiệm" in text:
        return {
            "experience_clean": text,
            "experience_min_years": 0,
            "experience_max_years": 0,
            "experience_type": "none_required",
        }

    if "dưới 1 năm" in text:
        return {
            "experience_clean": text,
            "experience_min_years": 0,
            "experience_max_years": 1,
            "experience_type": "range",
        }

    nums = [int(x) for x in re.findall(r"\d+", text)]

    if len(nums) >= 2 and any(k in text for k in ["-", "đến", "tới"]):
        return {
            "experience_clean": text,
            "experience_min_years": nums[0],
            "experience_max_years": nums[1],
            "experience_type": "range",
        }

    if len(nums) >= 1:
        if any(k in text for k in ["từ", "trên", "ít nhất", ">= "]):
            return {
                "experience_clean": text,
                "experience_min_years": nums[0],
                "experience_max_years": None,
                "experience_type": "minimum",
            }
        return {
            "experience_clean": text,
            "experience_min_years": nums[0],
            "experience_max_years": nums[0],
            "experience_type": "fixed",
        }

    return {
        "experience_clean": text,
        "experience_min_years": None,
        "experience_max_years": None,
        "experience_type": "unknown",
    }

In [26]:
exp_examples = pd.Series([
    "Không yêu cầu kinh nghiệm",
    "Dưới 1 năm",
    "Từ 2 năm",
    "2 - 4 năm",
    "Ít nhất 3 năm"
])

display(exp_examples.apply(parse_experience_range).apply(pd.Series))

,experience_clean,experience_min_years,experience_max_years,experience_type
0,không yêu cầu kinh nghiệm,0,0.0,none_required
1,dưới 1 năm,0,1.0,range
2,từ 2 năm,2,NaN,minimum
3,2 - 4 năm,2,4.0,range
4,ít nhất 3 năm,3,NaN,minimum


In [27]:
exp_parsed = df_clean["experience_raw"].apply(parse_experience_range)

exp_cols = [
    "experience_clean",
    "experience_min_years",
    "experience_max_years",
    "experience_type"
]

for col in exp_cols:
    df_clean[col] = exp_parsed.apply(lambda x: x[col])

display(
    df_clean[
        ["experience_raw"] + exp_cols
    ].head(10)
)

,experience_raw,experience_clean,experience_min_years,experience_max_years,experience_type
0,2 năm,2 năm,2,2.0,fixed
1,2 năm,2 năm,2,2.0,fixed
2,2 năm,2 năm,2,2.0,fixed
3,3 năm,3 năm,3,3.0,fixed
4,2 năm,2 năm,2,2.0,fixed
5,3 năm,3 năm,3,3.0,fixed
6,3 năm,3 năm,3,3.0,fixed
7,Không yêu cầu,không yêu cầu,0,0.0,none_required
8,1 năm,1 năm,1,1.0,fixed
9,1 năm,1 năm,1,1.0,fixed


In [28]:
EDUCATION_MAP = {
    "trung học": "high_school",
    "thpt": "high_school",
    "cao đẳng": "college",
    "đại học": "bachelor",
    "cử nhân": "bachelor",
    "bachelor": "bachelor",
    "thạc sĩ": "master",
    "master": "master",
    "tiến sĩ": "phd",
    "phd": "phd",
}

def normalize_education_level(text: str) -> str:
    text = clean_text_strict(text)
    if not text:
        return "unknown"

    for key, value in EDUCATION_MAP.items():
        if key in text:
            return value
    return "unknown"

In [29]:
df_clean["education_level_norm"] = df_clean["education_level_raw"].apply(normalize_education_level)

display(df_clean[["education_level_raw", "education_level_norm"]].head(10))

,education_level_raw,education_level_norm
0,Đại Học trở lên,bachelor
1,Đại Học trở lên,bachelor
2,Đại Học trở lên,bachelor
3,Đại Học trở lên,bachelor
4,Đại Học trở lên,bachelor
5,Đại Học trở lên,bachelor
6,Đại Học trở lên,bachelor
7,Đại Học trở lên,bachelor
8,Đại Học trở lên,bachelor
9,Đại Học trở lên,bachelor


In [30]:
EMPLOYMENT_TYPE_MAP = {
    "toàn thời gian": "full_time",
    "full time": "full_time",
    "full-time": "full_time",
    "bán thời gian": "part_time",
    "part time": "part_time",
    "part-time": "part_time",
    "thực tập": "internship",
    "intern": "internship",
    "internship": "internship",
    "freelance": "freelance",
    "cộng tác viên": "freelance",
    "hợp đồng": "contract",
    "contract": "contract",
    "remote": "remote",
    "hybrid": "hybrid",
}

def normalize_employment_type(text: str) -> str:
    text = clean_text_strict(text)
    if not text:
        return "unknown"

    for key, value in EMPLOYMENT_TYPE_MAP.items():
        if key in text:
            return value
    return "unknown"

In [31]:
df_clean["employment_type_norm"] = df_clean["employment_type_raw"].apply(normalize_employment_type)

display(df_clean[["employment_type_raw", "employment_type_norm"]].head(10))

,employment_type_raw,employment_type_norm
0,Toàn thời gian,full_time
1,Toàn thời gian,full_time
2,Toàn thời gian,full_time
3,Toàn thời gian,full_time
4,Toàn thời gian,full_time
5,Toàn thời gian,full_time
6,Toàn thời gian,full_time
7,Toàn thời gian,full_time
8,Toàn thời gian,full_time
9,Toàn thời gian,full_time


In [32]:
JOB_LEVEL_MAP = {
    "intern": "intern",
    "thực tập": "intern",
    "fresher": "fresher",
    "junior": "junior",
    "nhân viên": "junior",
    "chuyên viên": "junior",
    "mid": "middle",
    "middle": "middle",
    "senior": "senior",
    "trưởng nhóm": "lead",
    "team lead": "lead",
    "leader": "lead",
    "lead": "lead",
    "manager": "manager",
    "quản lý": "manager",
    "head": "manager",
    "director": "director",
}

def normalize_job_level(text: str, title_text: str = None) -> str:
    merged = " ".join([
        clean_text_strict(text),
        clean_text_strict(title_text)
    ]).strip()

    if not merged:
        return "unknown"

    for key, value in JOB_LEVEL_MAP.items():
        if key in merged:
            return value
    return "unknown"

In [33]:
df_clean["job_level_norm"] = df_clean.apply(
    lambda row: normalize_job_level(row.get("job_level_raw"), row.get("job_title_raw")),
    axis=1
)

display(
    df_clean[
        ["job_level_raw", "job_title_raw", "job_level_norm"]
    ].head(10)
)

,job_level_raw,job_title_raw,job_level_norm
0,Nhân viên,Junior FP&A Analyst - Hồ Chí Minh,junior
1,Nhân viên,Data Governance Specialist,junior
2,Nhân viên,Project Manager Dự Án AI HUB,junior
3,Nhân viên,AI Engineer,junior
4,Nhân viên,AI Engineer,junior
5,Nhân viên,Data Analyst,junior
6,Nhân viên,Data Engineer,junior
7,Trưởng nhóm,Fresher Data Engineer,fresher
8,Trưởng nhóm,Data Analyst Teamleader (Collection Analytics),lead
9,Trưởng nhóm,Data Analyst Teamleader (Collection Analytics) - Thu Nhập 50 Triệu Tại Hồ Chí Minh,lead


In [34]:
metadata_preview_cols = [
    "job_title_raw", "job_title_canonical", "job_family", "seniority_from_title",
    "location_raw", "location_city", "location_district", "location_norm", "work_mode",
    "salary_raw", "salary_min_vnd_month", "salary_max_vnd_month", "salary_type", "salary_is_negotiable",
    "experience_raw", "experience_min_years", "experience_max_years", "experience_type",
    "education_level_raw", "education_level_norm",
    "employment_type_raw", "employment_type_norm",
    "job_level_raw", "job_level_norm"
]
metadata_preview_cols = [c for c in metadata_preview_cols if c in df_clean.columns]

display(df_clean[metadata_preview_cols].head(10))

,job_title_raw,job_title_canonical,job_family,seniority_from_title,location_raw,location_city,location_district,location_norm,work_mode,salary_raw,salary_min_vnd_month,salary_max_vnd_month,salary_type,salary_is_negotiable,experience_raw,experience_min_years,experience_max_years,experience_type,education_level_raw,education_level_norm,employment_type_raw,employment_type_norm,job_level_raw,job_level_norm
0,Junior FP&A Analyst - Hồ Chí Minh,junior fp a analyst - hồ chí minh,other,junior,Hồ Chí Minh,hồ chí minh,huyện cũ tương ứng để dễ dàng tra cứu,hồ chí minh,unknown,12 - 20 triệu,12000000.0,20000000.0,range,0,2 năm,2,2.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior
1,Data Governance Specialist,data governance specialist,other,unknown,Hà Nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,unknown,20 - 30 triệu,20000000.0,30000000.0,range,0,2 năm,2,2.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior
2,Project Manager Dự Án AI HUB,project manager dự án ai hub,other,manager,Hà Nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,unknown,30 - 35 triệu,30000000.0,35000000.0,range,0,2 năm,2,2.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior
3,AI Engineer,ai engineer,data_science_ml,unknown,Hà Nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,unknown,Thoả thuận,NaN,NaN,unknown,1,3 năm,3,3.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior
4,AI Engineer,ai engineer,data_science_ml,unknown,Hà Nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,unknown,Thoả thuận,NaN,NaN,unknown,1,2 năm,2,2.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior
5,Data Analyst,data analyst,data_analytics,unknown,Hà Nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,unknown,Thoả thuận,NaN,NaN,unknown,1,3 năm,3,3.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior
6,Data Engineer,data engineer,data_engineering,unknown,Hà Nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,unknown,Thoả thuận,NaN,NaN,unknown,1,3 năm,3,3.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,junior
7,Fresher Data Engineer,fresher data engineer,data_engineering,junior,Hà Nội,hà nội,huyện cũ tương ứng để dễ dàng tra cứu,hà nội,unknown,6 - 9 triệu,6000000.0,9000000.0,range,0,Không yêu cầu,0,0.0,none_required,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Trưởng nhóm,fresher
8,Data Analyst Teamleader (Collection Analytics),data analyst teamleader,data_analytics,lead,Hồ Chí Minh,hồ chí minh,huyện cũ tương ứng để dễ dàng tra cứu,hồ chí minh,unknown,Thoả thuận,NaN,NaN,unknown,1,1 năm,1,1.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Trưởng nhóm,lead
9,Data Analyst Teamleader (Collection Analytics) - Thu Nhập 50 Triệu Tại Hồ Chí Minh,data analyst teamleader,data_analytics,lead,Hồ Chí Minh,hồ chí minh,huyện cũ tương ứng để dễ dàng tra cứu,hồ chí minh,unknown,Thoả thuận,NaN,NaN,unknown,1,1 năm,1,1.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Trưởng nhóm,lead


In [35]:
normalize_check_cols = [
    "job_title_canonical",
    "location_norm",
    "salary_min_vnd_month",
    "experience_min_years",
    "education_level_norm",
    "employment_type_norm",
    "job_level_norm"
]

report = {}
for col in normalize_check_cols:
    if col in df_clean.columns:
        if df_clean[col].dtype == "O":
            missing_ratio = df_clean[col].fillna("").astype(str).str.strip().eq("").mean()
        else:
            missing_ratio = df_clean[col].isna().mean()

        report[col] = {
            "missing_ratio": round(float(missing_ratio), 4),
            "nunique": int(df_clean[col].nunique(dropna=True))
        }

display(pd.DataFrame(report).T)

,missing_ratio,nunique
job_title_canonical,0.0000,229.0
location_norm,0.0000,8.0
salary_min_vnd_month,0.5662,32.0
experience_min_years,0.0000,6.0
education_level_norm,0.0000,4.0
employment_type_norm,0.0000,4.0
job_level_norm,0.0000,8.0


# 4. Normalize tags và skill extraction

In [36]:
def normalize_tags(text: str):
    text = normalize_empty_value(text)
    if text is None:
        return []

    text = clean_text_light(text)
    parts = re.split(r"[,;/|\n\-]+", text)
    parts = [p.strip().lower() for p in parts if p and p.strip()]
    return deduplicate_list(parts)

df_clean["tags_list"] = df_clean["tags_raw"].apply(normalize_tags)

display(df_clean[["tags_raw", "tags_list"]].head(10))

,tags_raw,tags_list
0,Chuyên môn Data Analyst; Tài chính; Kế toán,"[chuyên môn data analyst, tài chính, kế toán]"
1,Chuyên môn Data Analyst,[chuyên môn data analyst]
2,Chuyên môn IT Project Manager,[chuyên môn it project manager]
3,Chuyên môn AI Engineer; IT - Phần mềm,"[chuyên môn ai engineer, it, phần mềm]"
4,Chuyên môn AI Engineer,[chuyên môn ai engineer]
5,Chuyên môn Data Analyst,[chuyên môn data analyst]
6,Chuyên môn Data Engineer,[chuyên môn data engineer]
7,Chuyên môn Data Engineer; IT - Phần mềm,"[chuyên môn data engineer, it, phần mềm]"
8,Chuyên môn Data Analyst; Tài chính; Ngân hàng; IT - Phần mềm,"[chuyên môn data analyst, tài chính, ngân hàng, it, phần mềm]"
9,Chuyên môn Data Analyst; Tài chính; Ngân hàng,"[chuyên môn data analyst, tài chính, ngân hàng]"


In [37]:
SKILL_TAXONOMY = {
    "python": {"aliases": ["python", "python3"], "group": "programming"},
    "r": {"aliases": ["r"], "group": "programming"},
    "java": {"aliases": ["java"], "group": "programming"},
    "scala": {"aliases": ["scala"], "group": "programming"},
    "c++": {"aliases": ["c++"], "group": "programming"},
    "c#": {"aliases": ["c#"], "group": "programming"},

    "sql": {"aliases": ["sql", "mysql", "postgres", "postgresql", "oracle", "sql server"], "group": "database"},
    "mongodb": {"aliases": ["mongodb"], "group": "database"},

    "etl": {"aliases": ["etl", "elt"], "group": "data_engineering"},
    "data warehouse": {"aliases": ["data warehouse", "dwh"], "group": "data_engineering"},
    "data lake": {"aliases": ["data lake"], "group": "data_engineering"},
    "airflow": {"aliases": ["airflow"], "group": "data_engineering"},
    "dbt": {"aliases": ["dbt"], "group": "data_engineering"},
    "spark": {"aliases": ["spark", "apache spark"], "group": "data_engineering"},
    "hadoop": {"aliases": ["hadoop"], "group": "data_engineering"},

    "power bi": {"aliases": ["power bi", "powerbi", "pbi"], "group": "bi_visualization"},
    "tableau": {"aliases": ["tableau"], "group": "bi_visualization"},
    "looker": {"aliases": ["looker"], "group": "bi_visualization"},
    "excel": {"aliases": ["excel"], "group": "bi_visualization"},

    "statistics": {"aliases": ["statistics", "thống kê"], "group": "analytics"},
    "a/b testing": {"aliases": ["a/b testing", "ab testing", "a/b"], "group": "analytics"},
    "forecasting": {"aliases": ["forecast", "forecasting"], "group": "analytics"},

    "machine learning": {"aliases": ["machine learning", "ml"], "group": "ml_ai"},
    "deep learning": {"aliases": ["deep learning", "dl"], "group": "ml_ai"},
    "nlp": {"aliases": ["nlp", "natural language processing"], "group": "ml_ai"},
    "computer vision": {"aliases": ["computer vision", "cv"], "group": "ml_ai"},

    "aws": {"aliases": ["aws"], "group": "cloud"},
    "gcp": {"aliases": ["gcp", "google cloud"], "group": "cloud"},
    "azure": {"aliases": ["azure"], "group": "cloud"},

    "docker": {"aliases": ["docker"], "group": "deployment"},
    "kubernetes": {"aliases": ["kubernetes", "k8s"], "group": "deployment"},

    "data governance": {"aliases": ["data governance"], "group": "governance"},
    "data quality": {"aliases": ["data quality"], "group": "governance"},
    "data lineage": {"aliases": ["data lineage"], "group": "governance"},
}

In [38]:
def alias_to_regex(alias: str):
    alias = re.escape(alias.lower())
    if alias in {"r", "ml", "dl", "cv", "ai"}:
        return rf"(?<!\w){alias}(?!\w)"
    return rf"(?<!\w){alias}(?!\w)"

def extract_skills_from_text(text: str):
    text = clean_text_strict(text)
    if not text:
        return []

    found_skills = []
    for skill_name, meta in SKILL_TAXONOMY.items():
        aliases = meta.get("aliases", [])
        matched = False
        for alias in aliases:
            pattern = alias_to_regex(alias)
            if re.search(pattern, text, flags=re.IGNORECASE):
                matched = True
                break
        if matched:
            found_skills.append(skill_name)

    return deduplicate_list(found_skills)

def skill_to_group(skill_name: str) -> str:
    return SKILL_TAXONOMY.get(skill_name, {}).get("group", "other")

In [39]:
df_clean["skills_from_tags"] = df_clean["tags_raw"].apply(extract_skills_from_text)
df_clean["skills_from_title"] = df_clean["job_title_clean_light"].apply(extract_skills_from_text)
df_clean["skills_from_requirements"] = df_clean["requirements_clean_light"].apply(extract_skills_from_text)
df_clean["skills_from_description"] = df_clean["description_clean_light"].apply(extract_skills_from_text)

In [40]:
def merge_skill_lists(*lists):
    merged = []
    seen = set()
    for lst in lists:
        if not isinstance(lst, list):
            continue
        for item in lst:
            if item not in seen:
                seen.add(item)
                merged.append(item)
    return merged

In [41]:
df_clean["skills_extracted"] = df_clean.apply(
    lambda row: merge_skill_lists(
        row["skills_from_tags"],
        row["skills_from_title"],
        row["skills_from_requirements"],
        row["skills_from_description"]
    ),
    axis=1
)

display(
    df_clean[
        [
            "job_title_raw",
            "tags_raw",
            "skills_from_tags",
            "skills_from_title",
            "skills_from_requirements",
            "skills_extracted"
        ]
    ].head(10)
)

,job_title_raw,tags_raw,skills_from_tags,skills_from_title,skills_from_requirements,skills_extracted
0,Junior FP&A Analyst - Hồ Chí Minh,Chuyên môn Data Analyst; Tài chính; Kế toán,[],[],[],[]
1,Data Governance Specialist,Chuyên môn Data Analyst,[],[data governance],"[sql, etl, data warehouse, data governance]","[data governance, sql, etl, data warehouse, data quality]"
2,Project Manager Dự Án AI HUB,Chuyên môn IT Project Manager,[],[],[],[]
3,AI Engineer,Chuyên môn AI Engineer; IT - Phần mềm,[],[],"[python, machine learning]","[python, machine learning]"
4,AI Engineer,Chuyên môn AI Engineer,[],[],[],[]
5,Data Analyst,Chuyên môn Data Analyst,[],[],"[sql, etl, data warehouse, machine learning]","[sql, etl, data warehouse, machine learning]"
6,Data Engineer,Chuyên môn Data Engineer,[],[],"[sql, etl]","[sql, etl]"
7,Fresher Data Engineer,Chuyên môn Data Engineer; IT - Phần mềm,[],[],"[python, sql, etl, spark]","[python, sql, etl, spark, power bi, aws]"
8,Data Analyst Teamleader (Collection Analytics),Chuyên môn Data Analyst; Tài chính; Ngân hàng; IT - Phần mềm,[],[],"[python, sql]","[python, sql, data warehouse]"
9,Data Analyst Teamleader (Collection Analytics) - Thu Nhập 50 Triệu Tại Hồ Chí Minh,Chuyên môn Data Analyst; Tài chính; Ngân hàng,[],[],"[python, sql]","[python, sql, data warehouse]"


In [42]:
df_clean["skill_groups"] = df_clean["skills_extracted"].apply(
    lambda xs: deduplicate_list([skill_to_group(x) for x in xs]) if isinstance(xs, list) else []
)

df_clean["skills_text"] = df_clean["skills_extracted"].apply(
    lambda x: "; ".join(x) if isinstance(x, list) else ""
)

df_clean["skill_groups_text"] = df_clean["skill_groups"].apply(
    lambda x: "; ".join(x) if isinstance(x, list) else ""
)

In [43]:
df_clean["num_skills"] = df_clean["skills_extracted"].apply(lambda x: len(x) if isinstance(x, list) else 0)
df_clean["num_skill_groups"] = df_clean["skill_groups"].apply(lambda x: len(x) if isinstance(x, list) else 0)

display(df_clean[["num_skills", "num_skill_groups"]].describe())
print("Avg skills per job:", round(df_clean["num_skills"].mean(), 2))

,num_skills,num_skill_groups
count,325.000000,325.000000
mean,3.923077,2.421538
std,3.216027,1.784059
min,0.000000,0.000000
25%,1.000000,1.000000
50%,4.000000,2.000000
75%,6.000000,4.000000
max,15.000000,7.000000


Avg skills per job: 3.92


In [44]:
all_skills = df_clean["skills_extracted"].explode().dropna().tolist()
skill_counts = Counter(all_skills)
top_skills = pd.DataFrame(skill_counts.most_common(30), columns=["skill", "count"])
display(top_skills)

,skill,count
0,python,148
1,sql,143
2,machine learning,99
3,power bi,62
4,etl,55
5,statistics,53
6,excel,48
7,tableau,47
8,data warehouse,46
9,spark,45


In [45]:
empty_skill_ratio = (df_clean["num_skills"] == 0).mean()
print(f"Tỷ lệ job không extract được skill: {round(empty_skill_ratio, 4)}")

Tỷ lệ job không extract được skill: 0.2


In [46]:
def summarize_text(text: str, max_chars: int = 500) -> str:
    text = safe_str(text)
    if not text:
        return ""
    return text[:max_chars].strip()

def build_job_text_title(row):
    return safe_str(row.get("job_title_canonical") or row.get("job_title_display"))

def build_job_text_skills(row):
    parts = []
    if row.get("skills_text"):
        parts.append(row["skills_text"])
    if row.get("skill_groups_text"):
        parts.append(row["skill_groups_text"])
    return " ; ".join([p for p in parts if p])

def build_job_text_requirements(row):
    req = summarize_text(row.get("requirements_clean_light"), 600)
    exp = row.get("experience_min_years")
    edu = row.get("education_level_norm")
    pieces = []
    if req:
        pieces.append(req)
    if exp is not None:
        pieces.append(f"minimum experience {int(exp)} years")
    if edu and edu != "unknown":
        pieces.append(f"education {edu}")
    return " . ".join(pieces)

def build_job_text_description(row):
    return summarize_text(row.get("description_clean_light"), 600)

def build_job_text_semantic(row):
    parts = [
        build_job_text_title(row),
        build_job_text_skills(row),
        build_job_text_requirements(row),
        build_job_text_description(row),
    ]
    return "\n".join([p for p in parts if p])

def build_job_text_structured(row):
    parts = []
    if row.get("job_title_canonical"):
        parts.append(f"[TITLE] {row['job_title_canonical']}")
    if row.get("skills_text"):
        parts.append(f"[SKILLS] {row['skills_text']}")
    if row.get("skill_groups_text"):
        parts.append(f"[SKILL_GROUPS] {row['skill_groups_text']}")
    if row.get("experience_min_years") is not None:
        parts.append(f"[EXPERIENCE] {row['experience_min_years']}")
    if row.get("education_level_norm") and row["education_level_norm"] != "unknown":
        parts.append(f"[EDUCATION] {row['education_level_norm']}")
    if row.get("location_norm"):
        parts.append(f"[LOCATION] {row['location_norm']}")
    if row.get("work_mode") and row["work_mode"] != "unknown":
        parts.append(f"[WORK_MODE] {row['work_mode']}")
    if row.get("job_level_norm") and row["job_level_norm"] != "unknown":
        parts.append(f"[LEVEL] {row['job_level_norm']}")
    return "\n".join(parts)

def build_job_text_chatbot(row):
    parts = []
    if row.get("job_title_display"):
        parts.append(f"Job title: {row['job_title_display']}")
    if row.get("job_family"):
        parts.append(f"Job family: {row['job_family']}")
    if row.get("requirements_clean_struct"):
        parts.append(f"Requirements:\n{summarize_text(row['requirements_clean_struct'], 1200)}")
    if row.get("description_clean_struct"):
        parts.append(f"Responsibilities:\n{summarize_text(row['description_clean_struct'], 1200)}")
    if row.get("benefits_clean_struct"):
        parts.append(f"Benefits:\n{summarize_text(row['benefits_clean_struct'], 800)}")
    if row.get("skills_text"):
        parts.append(f"Important skills: {row['skills_text']}")
    return "\n\n".join(parts)

In [47]:
df_clean["job_text_title"] = df_clean.apply(build_job_text_title, axis=1)
df_clean["job_text_skills"] = df_clean.apply(build_job_text_skills, axis=1)
df_clean["job_text_requirements"] = df_clean.apply(build_job_text_requirements, axis=1)
df_clean["job_text_description"] = df_clean.apply(build_job_text_description, axis=1)

df_clean["job_text_semantic"] = df_clean.apply(build_job_text_semantic, axis=1)
df_clean["job_text_structured"] = df_clean.apply(build_job_text_structured, axis=1)
df_clean["job_text_chatbot"] = df_clean.apply(build_job_text_chatbot, axis=1)

print("[INFO] Đã tạo xong job_text_semantic / structured / chatbot")

[INFO] Đã tạo xong job_text_semantic / structured / chatbot


In [48]:
display(
    df_clean[
        [
            "job_title_raw",
            "job_title_canonical",
            "skills_text",
            "job_text_title",
            "job_text_skills",
            "job_text_requirements",
            "job_text_semantic"
        ]
    ].head(3)
)

,job_title_raw,job_title_canonical,skills_text,job_text_title,job_text_skills,job_text_requirements,job_text_semantic
0,Junior FP&A Analyst - Hồ Chí Minh,junior fp a analyst - hồ chí minh,,junior fp a analyst - hồ chí minh,,"- At least 2 year experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor s degree in Finance/Accounting,...",junior fp a analyst - hồ chí minh\n- At least 2 year experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bache...
1,Data Governance Specialist,data governance specialist,data governance; sql; etl; data warehouse; data quality,data governance specialist,data governance; sql; etl; data warehouse; data quality ; governance; database; data_engineering,"- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","data governance specialist\ndata governance; sql; etl; data warehouse; data quality ; governance; database; data_engineering\n- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thôn..."
2,Project Manager Dự Án AI HUB,project manager dự án ai hub,,project manager dự án ai hub,,"Tối thiểu 2-3 năm kinh nghiệm ở vị trí Product Manager, Project Manager hoặc Operations Manager. Ưu tiên ứng viên từng tham gia các dự án automation, AI, data analytics hoặc chuyển đổi số. Ưu tiên...","project manager dự án ai hub\nTối thiểu 2-3 năm kinh nghiệm ở vị trí Product Manager, Project Manager hoặc Operations Manager. Ưu tiên ứng viên từng tham gia các dự án automation, AI, data analyti..."


In [49]:
for col in ["job_text_title", "job_text_skills", "job_text_requirements", "job_text_description", "job_text_semantic", "job_text_chatbot"]:
    df_clean[f"{col}_len"] = df_clean[col].fillna("").str.len()

display(
    df_clean[
        [f"{c}_len" for c in ["job_text_title", "job_text_skills", "job_text_requirements", "job_text_description", "job_text_semantic", "job_text_chatbot"]]
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
job_text_title_len,325.0,27.033846,17.485323,8.0,14.0,23.0,34.0,151.0
job_text_skills_len,325.0,64.163077,48.991507,0.0,23.0,64.0,99.0,199.0
job_text_requirements_len,325.0,578.883077,119.135201,168.0,535.0,649.0,650.0,653.0
job_text_description_len,325.0,544.935385,103.180856,149.0,535.0,600.0,600.0,600.0
job_text_semantic_len,325.0,1217.815385,212.747336,422.0,1143.0,1295.0,1359.0,1513.0
job_text_chatbot_len,325.0,2324.898462,669.624212,793.0,1844.0,2371.0,2850.0,3448.0


In [50]:
print("=== VERY SHORT SEMANTIC TEXT ===")
display(
    df_clean[df_clean["job_text_semantic_len"] < 50][
        ["job_title_raw", "job_text_semantic"]
    ].head(5)
)

print("=== VERY LONG CHATBOT TEXT ===")
display(
    df_clean[df_clean["job_text_chatbot_len"] > 2000][
        ["job_title_raw", "job_text_chatbot"]
    ].head(5)
)

=== VERY SHORT SEMANTIC TEXT ===


,job_title_raw,job_text_semantic


=== VERY LONG CHATBOT TEXT ===


,job_title_raw,job_text_chatbot
0,Junior FP&A Analyst - Hồ Chí Minh,Job title: Junior FP A Analyst - Hồ Chí Minh\n\nJob family: other\n\nRequirements:\n- At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit servi...
1,Data Governance Specialist,"Job title: Data Governance Specialist\n\nJob family: other\n\nRequirements:\n- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh v..."
2,Project Manager Dự Án AI HUB,"Job title: Project Manager Dự Án AI HUB\n\nJob family: other\n\nRequirements:\nTối thiểu 2-3 năm kinh nghiệm ở vị trí Product Manager, Project Manager hoặc Operations Manager. Ưu tiên ứng viên từn..."
3,AI Engineer,Job title: AI Engineer\n\nJob family: data_science_ml\n\nRequirements:\nKinh nghiệm AI/ML ứng dụng thực tế Python và xử lý dữ liệu tốt Tư duy production và tối ưu chi phí Ưu tiên ứng viên đã có ki...
4,AI Engineer,"Job title: AI Engineer\n\nJob family: data_science_ml\n\nRequirements:\nTốt nghiệp Đại học chuyên ngành CNTT, Khoa học máy tính, Điện tử viễn thông,... Từ 2 năm kinh nghiệm xây dựng và triển khai ..."


# 5. PhoBERT section-aware matching

In [51]:
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "vinai/phobert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print("[INFO] PhoBERT loaded on", device)

d:\TTTN\Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 39147.62it/s]
RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.bias                    | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] PhoBERT loaded on cpu


In [52]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, dim=1) / torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)

In [53]:
def encode_texts(texts, batch_size=16, max_length=256):
    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = [safe_str(x) if safe_str(x) else "[EMPTY]" for x in texts[i:i+batch_size]]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        with torch.no_grad():
            model_output = model(input_ids=input_ids, attention_mask=attention_mask)

        batch_embeddings = mean_pooling(model_output, attention_mask)
        batch_embeddings = torch.nn.functional.normalize(batch_embeddings, p=2, dim=1)

        embeddings.append(batch_embeddings.cpu().numpy())

    return np.vstack(embeddings)

In [54]:
job_title_embeddings = encode_texts(df_clean["job_text_title"].fillna("").tolist(), batch_size=16, max_length=64)
job_skills_embeddings = encode_texts(df_clean["job_text_skills"].fillna("").tolist(), batch_size=16, max_length=96)
job_requirements_embeddings = encode_texts(df_clean["job_text_requirements"].fillna("").tolist(), batch_size=16, max_length=192)
job_description_embeddings = encode_texts(df_clean["job_text_description"].fillna("").tolist(), batch_size=16, max_length=192)

print("job_title_embeddings      :", job_title_embeddings.shape)
print("job_skills_embeddings     :", job_skills_embeddings.shape)
print("job_requirements_embeddings:", job_requirements_embeddings.shape)
print("job_description_embeddings :", job_description_embeddings.shape)

100%|██████████| 21/21 [00:52<00:00,  2.49s/it]

job_title_embeddings      : (325, 768)
job_skills_embeddings     : (325, 768)
job_requirements_embeddings: (325, 768)
job_description_embeddings : (325, 768)


In [55]:
df_clean["emb_title"] = list(job_title_embeddings)
df_clean["emb_skills"] = list(job_skills_embeddings)
df_clean["emb_requirements"] = list(job_requirements_embeddings)
df_clean["emb_description"] = list(job_description_embeddings)

print("[INFO] Saved section embeddings into dataframe")

[INFO] Saved section embeddings into dataframe


In [56]:
query_profile = {
    "target_title": "data scientist",
    "skills": ["python", "sql", "power bi", "excel", "docker"],
    "skill_groups": ["programming", "database", "bi_visualization", "deployment"],
    "experience_min_years": 3,
    "education_level": "bachelor",
    "preferred_location": "hồ chí minh",
    "work_mode": "unknown",
    "summary": "phân tích dữ liệu, xây dựng dashboard, huấn luyện mô hình machine learning, xử lý dữ liệu lớn"
}

query_text_title = query_profile["target_title"]
query_text_skills = "; ".join(query_profile["skills"] + query_profile["skill_groups"])
query_text_requirements = f"minimum experience {query_profile['experience_min_years']} years. education {query_profile['education_level']}"
query_text_description = query_profile["summary"]

print(query_text_title)
print(query_text_skills)
print(query_text_requirements)
print(query_text_description)

data scientist
python; sql; power bi; excel; docker; programming; database; bi_visualization; deployment
minimum experience 3 years. education bachelor
phân tích dữ liệu, xây dựng dashboard, huấn luyện mô hình machine learning, xử lý dữ liệu lớn


In [57]:
query_emb_title = encode_texts([query_text_title], batch_size=1, max_length=64)[0]
query_emb_skills = encode_texts([query_text_skills], batch_size=1, max_length=96)[0]
query_emb_requirements = encode_texts([query_text_requirements], batch_size=1, max_length=192)[0]
query_emb_description = encode_texts([query_text_description], batch_size=1, max_length=192)[0]

print("Query embeddings ready.")

100%|██████████| 1/1 [00:00<00:00, 13.38it/s]

Query embeddings ready.


In [58]:
def cosine_similarity(vec, mat):
    return np.dot(mat, vec)

In [59]:
df_clean["score_title_sem"] = cosine_similarity(query_emb_title, job_title_embeddings)
df_clean["score_skills_sem"] = cosine_similarity(query_emb_skills, job_skills_embeddings)
df_clean["score_requirements_sem"] = cosine_similarity(query_emb_requirements, job_requirements_embeddings)
df_clean["score_description_sem"] = cosine_similarity(query_emb_description, job_description_embeddings)

df_clean["semantic_score"] = (
    0.40 * df_clean["score_title_sem"] +
    0.30 * df_clean["score_skills_sem"] +
    0.20 * df_clean["score_requirements_sem"] +
    0.10 * df_clean["score_description_sem"]
)

In [60]:
top_k = 10

top_jobs_semantic = df_clean.sort_values("semantic_score", ascending=False).head(top_k)

display(
    top_jobs_semantic[
        [
            "job_title_raw",
            "job_title_canonical",
            "skills_text",
            "location_norm",
            "semantic_score"
        ]
    ]
)

,job_title_raw,job_title_canonical,skills_text,location_norm,semantic_score
183,Data Scientist (Middle+),data scientist,python; sql; airflow; dbt; machine learning; gcp,hà nội,0.890179
186,Data Scientist,data scientist,python; r; sql; power bi; tableau; statistics; machine learning; data governance; data quality,hồ chí minh,0.890085
185,Data Scientist,data scientist,python; sql; airflow; machine learning; deep learning; docker; kubernetes,hồ chí minh,0.871176
187,Data Scientist,data scientist,python; sql; airflow; machine learning; deep learning; docker; kubernetes,hà nội,0.869978
291,Senior Data Scientist (Business Machine Learning - HCM),senior data scientist,machine learning; python; r; sql; mongodb; spark; hadoop; statistics; nlp; aws,hồ chí minh,0.819703
142,Data Analyst,data analyst,python; r; sql; power bi; tableau; statistics,hà nội,0.806369
148,Data Analyst,data analyst,sql; spark; hadoop; power bi; tableau,hà nội,0.802034
144,Data Analyst,data analyst,python; sql; power bi; tableau; looker,hà nội,0.800853
182,Data Scientist (Middle+) - Salary Upto 35M,data scientist - salary upto 35m,python; sql; airflow; dbt; machine learning; gcp,hà nội,0.800849
253,Nhân Viên Phân Tích Dữ Liệu,data analyst,python; power bi; tableau; excel,hải phòng,0.799883


In [61]:
def compute_overlap_score(source_list, target_list):
    if not source_list or not target_list:
        return 0.0
    s1 = set(source_list)
    s2 = set(target_list)
    if not s2:
        return 0.0
    return len(s1 & s2) / len(s2)

def compute_title_score(query_title: str, job_title: str) -> float:
    q = clean_text_strict(query_title)
    j = clean_text_strict(job_title)
    if not q or not j:
        return 0.0
    if q == j:
        return 1.0
    if q in j or j in q:
        return 0.8
    q_tokens = set(q.split())
    j_tokens = set(j.split())
    if not j_tokens:
        return 0.0
    return len(q_tokens & j_tokens) / len(j_tokens)

def compute_experience_score(query_exp, job_exp_min, job_exp_max):
    if query_exp is None or pd.isna(query_exp):
        return 0.5
    if job_exp_min is None and job_exp_max is None:
        return 0.5
    if job_exp_min is not None and query_exp < job_exp_min:
        return 0.0
    if job_exp_max is not None and query_exp > job_exp_max + 2:
        return 0.5
    return 1.0

def compute_location_score(query_location: str, job_location: str) -> float:
    q = clean_text_strict(query_location)
    j = clean_text_strict(job_location)
    if not q or q == "unknown":
        return 0.5
    if not j:
        return 0.3
    if q == j or q in j or j in q:
        return 1.0
    return 0.0

query_skills = query_profile["skills"]
query_skill_groups = query_profile["skill_groups"]
query_exp = query_profile["experience_min_years"]
query_location = query_profile["preferred_location"]
query_title = query_profile["target_title"]

df_clean["skill_exact_score"] = df_clean["skills_extracted"].apply(
    lambda x: compute_overlap_score(query_skills, x)
)

df_clean["skill_group_score"] = df_clean["skill_groups"].apply(
    lambda x: compute_overlap_score(query_skill_groups, x)
)

df_clean["title_score"] = df_clean["job_title_canonical"].apply(
    lambda x: compute_title_score(query_title, x)
)

df_clean["experience_score"] = df_clean.apply(
    lambda row: compute_experience_score(
        query_exp,
        row.get("experience_min_years"),
        row.get("experience_max_years")
    ),
    axis=1
)

df_clean["location_score"] = df_clean["location_norm"].apply(
    lambda x: compute_location_score(query_location, x)
)

In [62]:
df_clean["final_score"] = (
    0.40 * df_clean["semantic_score"] +
    0.20 * df_clean["skill_exact_score"] +
    0.10 * df_clean["skill_group_score"] +
    0.15 * df_clean["title_score"] +
    0.10 * df_clean["experience_score"] +
    0.05 * df_clean["location_score"]
)

df_clean["final_score"] = df_clean["final_score"].fillna(0.0)

In [63]:
top_jobs = df_clean.sort_values("final_score", ascending=False).head(10)

display(
    top_jobs[
        [
            "job_title_raw",
            "job_title_canonical",
            "skills_text",
            "skill_groups_text",
            "location_norm",
            "semantic_score",
            "skill_exact_score",
            "skill_group_score",
            "title_score",
            "experience_score",
            "location_score",
            "final_score"
        ]
    ]
)

,job_title_raw,job_title_canonical,skills_text,skill_groups_text,location_norm,semantic_score,skill_exact_score,skill_group_score,title_score,experience_score,location_score,final_score
185,Data Scientist,data scientist,python; sql; airflow; machine learning; deep learning; docker; kubernetes,programming; database; data_engineering; ml_ai; deployment,hồ chí minh,0.871176,0.428571,0.6,1.000000,1.0,1.0,0.794184
140,Data Analyst (Lĩnh Vực EdTech),data analyst,sql; power bi; tableau; excel,database; bi_visualization,hồ chí minh,0.786249,0.750000,1.0,0.500000,1.0,1.0,0.789500
99,Chuyên Viên Phân Tích Dữ Liệu,data analyst,sql; excel,database; bi_visualization,hà nội,0.784022,1.000000,1.0,0.500000,1.0,0.0,0.788609
186,Data Scientist,data scientist,python; r; sql; power bi; tableau; statistics; machine learning; data governance; data quality,programming; database; bi_visualization; analytics; ml_ai; governance,hồ chí minh,0.890085,0.333333,0.5,1.000000,1.0,1.0,0.772701
143,Data Analyst,data analyst,sql,database,hà nội,0.718238,1.000000,1.0,0.500000,1.0,0.0,0.762295
96,Chuyên Viên Phân Tích Dữ Liệu (Hàng May Mặc),data analyst,excel,bi_visualization,hưng yên,0.712402,1.000000,1.0,0.500000,1.0,0.0,0.759961
133,Data Analyst ( Chuyên Viên Phân Tích Dữ Liệu ),data analyst,python; sql; power bi; tableau; excel,programming; database; bi_visualization,hà nội,0.793445,0.800000,1.0,0.500000,1.0,0.0,0.752378
71,Business Analyst (Contact Center),business analyst,sql; excel,database; bi_visualization,hồ chí minh,0.738281,1.000000,1.0,0.000000,1.0,1.0,0.745313
187,Data Scientist,data scientist,python; sql; airflow; machine learning; deep learning; docker; kubernetes,programming; database; data_engineering; ml_ai; deployment,hà nội,0.869978,0.428571,0.6,1.000000,1.0,0.0,0.743706
138,Data Analyst Intern,data analyst intern,excel,bi_visualization,hồ chí minh,0.732429,1.000000,1.0,0.333333,0.5,1.0,0.742972


In [64]:
matching_cols = [
    "job_url",
    "job_title_raw",
    "job_title_display",
    "job_title_canonical",
    "job_family",
    "seniority_from_title",
    "location_norm",
    "location_city",
    "location_district",
    "work_mode",
    "salary_min_vnd_month",
    "salary_max_vnd_month",
    "salary_type",
    "salary_is_negotiable",
    "experience_min_years",
    "experience_max_years",
    "experience_type",
    "education_level_norm",
    "employment_type_norm",
    "job_level_norm",
    "skills_extracted",
    "skill_groups",
    "job_text_title",
    "job_text_skills",
    "job_text_requirements",
    "job_text_description",
    "job_text_semantic",
    "job_text_structured",
]

chatbot_cols = [
    "job_url",
    "job_title_display",
    "job_title_canonical",
    "job_family",
    "skills_extracted",
    "skill_groups",
    "requirements_clean_struct",
    "description_clean_struct",
    "benefits_clean_struct",
    "job_text_chatbot",
]

df_matching_ready = df_clean[[c for c in matching_cols if c in df_clean.columns]].copy()
df_chatbot_ready = df_clean[[c for c in chatbot_cols if c in df_clean.columns]].copy()

matching_path = ARTIFACT_DIR / "jobs_matching_ready.parquet"
chatbot_path = ARTIFACT_DIR / "jobs_chatbot_ready.parquet"

df_matching_ready.to_parquet(matching_path, index=False)
df_chatbot_ready.to_parquet(chatbot_path, index=False)

np.save(ARTIFACT_DIR / "job_title_embeddings.npy", job_title_embeddings)
np.save(ARTIFACT_DIR / "job_skills_embeddings.npy", job_skills_embeddings)
np.save(ARTIFACT_DIR / "job_requirements_embeddings.npy", job_requirements_embeddings)
np.save(ARTIFACT_DIR / "job_description_embeddings.npy", job_description_embeddings)

print("[INFO] Saved:")
print(" -", matching_path)
print(" -", chatbot_path)

[INFO] Saved:
 - D:\TTTN\Project\scripts\outputs\artifacts\jobs_matching_ready.parquet
 - D:\TTTN\Project\scripts\outputs\artifacts\jobs_chatbot_ready.parquet
